In [1]:
import pandas as pd
import numpy as np

# Methodology

Sentiment labels are assigned to each news article based on the sign of this aggregated `three-day excess return`. This excess return is calculated from the day a news article is first published and extends over the two subsequent days. To elaborate, excess return is defined as the difference between the return of a particular stock and the overall market return on the same day. This calculation is not limited to the day the news is published; instead, it aggregates the returns for the following two days as well, providing a
comprehensive three-day outlook.

A positive aggregated excess return leads to a sentiment label of `1`, indicating a positive sentiment. Conversely, a non-positive aggregated excess return results in a sentiment label of `0`, suggesting a negative sentiment.

In [9]:
stocks = pd.ExcelFile("./Data/stocks_data.xlsx")

stocks_dict = {
    sheet_name: stocks.parse(sheet_name) for sheet_name in stocks.sheet_names
}

In [34]:
def excess_return_sentiment_label(stock_data, market_data, num_days=3, price_used='Adj Close'):
    # Make a copy so as not to modify original
    stock_df = stock_data.copy()
    market_df = market_data.copy()

    # Set date as index
    stock_df = stock_df.set_index('date')
    market_df = market_df.set_index('date')

    # Filter for necessary price columns
    stock_df = stock_df[[price_used]].copy()
    market_df = market_df[[price_used]].copy()
    market_df = market_df.add_prefix('SPY_')

    # Calculate daily returns
    merged_df = stock_df.join(market_df)
    merged_df = merged_df.pct_change()

    # Calculate specified X-days aggregated returns
    merged_df['excess_returns'] = merged_df['Adj Close'] - merged_df['SPY_Adj Close']
    merged_df['X_days_excess_returns'] = merged_df['excess_returns'].rolling(num_days).sum().shift(-(num_days-1))
    
    # Create sentiment label
    merged_df['sentiment_label'] = np.sign(merged_df['X_days_excess_returns'])
    
    return merged_df

In [28]:
news = pd.ExcelFile('./Data/news_data.xlsx')
news_dict = {
    sheet_name: news.parse(sheet_name) for sheet_name in news.sheet_names
}

In [58]:
labelled_news_dict = {}

for k, v in news_dict.items():

    try:
        # Prepare the excess_returns dataframe
        excess_returns_df = excess_return_sentiment_label(
            stocks_dict[k], stocks_dict["SPY"]
        )

        # Set date column of news dataframe to datetime
        temp_df = v.copy()
        temp_df["date"] = pd.to_datetime(temp_df["date"]).dt.normalize()
        temp_df["sentiment_label"] = temp_df["date"].apply(
            lambda x: (
                excess_returns_df.loc[x]["sentiment_label"]
                if x in excess_returns_df.index
                else np.nan
            )
        )

        labelled_news_dict[k] = temp_df
        
    except:
        pass

In [60]:
labelled_news_dict['AAPL']

,date,title,source,sentiment_label
0,2024-11-29,Apple (AAPL) technical analysis - Apple (NASDA...,https://news.google.com/rss/articles/CBMiygFBV...,NaN
1,2024-12-01,Apple Inc. (AAPL) Gains Momentum with iPhone 1...,https://news.google.com/rss/articles/CBMigAFBV...,NaN
2,2024-11-29,Apple Stock: Potential Breakout Ahead (Technic...,https://news.google.com/rss/articles/CBMisAFBV...,NaN
3,2024-11-29,Apple Inc. (AAPL) Is a Trending Stock: Facts t...,https://news.google.com/rss/articles/CBMigAFBV...,NaN
4,2024-12-01,It Costs 10 Times More To Protect This Billion...,https://news.google.com/rss/articles/CBMizwFBV...,NaN
...,...,...,...,...
94,2024-11-05,Here is What to Know Beyond Why Apple Inc. (AA...,https://news.google.com/rss/articles/CBMie0FVX...,-1.0
95,2024-11-28,Advocate Group LLC Has $11.13 Million Stake in...,https://news.google.com/rss/articles/CBMiuwFBV...,NaN
96,2024-10-19,Apple Inc. (AAPL): Among the Most Owned Stocks...,https://news.google.com/rss/articles/CBMie0FVX...,NaN
97,2024-11-28,"Planned Solutions Inc. Purchases 3,472 Shares ...",https://news.google.com/rss/articles/CBMivgFBV...,NaN
